# Phase 11: Smart Context-Aware Routing Engine Analysis
### AI-Driven Intelligent Greywater Management and Smart Reuse Routing System

---
### Scientific & Regulatory Disclaimer
> **"The routing engine is a decision-support system. Its recommendations do not constitute regulatory approval or a guarantee of water safety."**
>
> **"Because the supervised routing labels are rule-derived, model performance measures agreement with the operational labeling framework rather than independent real-world ground truth."**



## 1. Objective

Phase 11 implements the **Central Context-Aware Routing Engine** of the intelligent greywater recycling facility.
It synthesizes multiple disparate signals into an auditable, deterministic operational routing decision:
1. Supervised Machine Learning classification (Optimized XGBoost)
2. Unsupervised Statistical Anomaly Detection (Isolation Forest)
3. Deterministic Water-Quality Safety Cutoffs (acute biological/chemical screening)
4. Meteorological & Environmental Context (surface precipitation, precipitation forecast, freezing risk)
5. Physical Storage State & Biochemical Deterioration Kinetics (tank mass balance, stagnation risk)
6. Operational Constraints & Operator Discretion



In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import routing
from routing.decision_schema import (
    RoutingInput,
    ROUTE_SEWER_BYPASS,
    ROUTE_BIO_FILTRATION,
    ROUTE_RESTRICTED_IRRIGATION,
    ROUTE_INDOOR_REUSE
)
from routing.routing_engine import SmartRoutingEngine
from routing.inference_pipeline import ContextAwareRoutingPipeline
from routing.run_routing_simulation import run_synthetic_routing_scenarios, run_batch_test_evaluation

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 4.5)
plt.rcParams['font.size'] = 10

print("Smart Routing Subsystem and Pipeline successfully initialized.")



## 2. Decision Hierarchy Architecture

Decisions are governed by a strict hierarchical precedence matrix:
```
Priority 1: CRITICAL WATER-QUALITY SAFETY (Acute toxic, pathogen, or septic hazard)
      ↓
Priority 2: HIGH-RISK ANOMALY (Statistical outlier + independent water quality violation)
      ↓
Priority 3: MODEL ROUTING PREDICTION (Supervised XGBoost classification)
      ↓
Priority 4: STORAGE / DETERIORATION CONDITION (Fluid age, stagnation, decay index)
      ↓
Priority 5: WEATHER / ENVIRONMENTAL CONTEXT (Precipitation, freeze, saturation)
      ↓
Priority 6: NORMAL OPERATIONAL PREFERENCE (Dispatch / pumping action)
```
*Higher priorities strictly supersede lower priorities. Favorable weather can NEVER override a water quality safety condition.*



## 3. Supervised Model Prediction (XGBoost)

We initialize the `ContextAwareRoutingPipeline`, which wraps the serialized `models/xgboost_optimized.pkl` and `models/isolation_forest.pkl`.



In [ ]:
pipeline = ContextAwareRoutingPipeline()
print("Model artifacts loaded successfully:")
print(f"  XGBoost estimators:       {pipeline.xgb_model.n_estimators}")
print(f"  Isolation Forest trees:   {pipeline.iso_model.n_estimators}")
print(f"  Expected feature vector:  {len(pipeline.expected_features)} features")



## 4. Isolation Forest Anomaly Detection

Isolation Forest acts as an independent statistical screener.
- **`ANOMALY_CLEAN`**: Observation lies within standard multidimensional training distribution.
- **`ANOMALY_REVIEW`**: Observation diverges statistically, but lacks acute physical toxicity. Retains ML candidate route under advisory review.
- **`ANOMALY_SAFETY_OVERRIDE`**: Statistical divergence confirmed with physical hazard threshold breach $\rightarrow$ direct Sewer Bypass.



## 5. Deterministic Safety Screening Layer

The `WaterQualitySafetyEngine` evaluates all incoming streams against absolute microbiological, chemical, and physical cutoffs derived from WHO, US EPA, and NSF 350 standards:
- Pathogen Hazard: $E. coli \ge 500,000\text{ CFU/100mL}$
- Organic Hazard: $COD \ge 800\text{ mg/L}$, $BOD \ge 400\text{ mg/L}$
- Septic Condition: $DO < 1.0\text{ mg/L}$ with high organics ($BOD \ge 300$ or $COD \ge 600$)
- Optical Scattering: $Turbidity \ge 140\text{ NTU}$, $TSS \ge 250\text{ mg/L}$
- Corrosive/Caustic Dump: $pH < 6.0$ or $pH > 9.0$



## 6. Weather & Environmental Context Arbitration

- **Restricted Irrigation**: Active rain $\ge 5\text{ mm}$ or forecast probability $\ge 50\%$ defers application (`STORE_FOR_LATER` / `DEFER_ROUTE`) to prevent runoff and ponding.
- **Indoor Reuse**: Hydraulically decoupled; toilet flushing remains `ALLOW_ROUTE`.
- **Bio-filtration**: Monitored for torrential storm washouts ($\ge 25\text{ mm}$).
- **Sewer Bypass**: Strict health cutoff; favorable weather **CANNOT** override.



## 7. Storage Tank & Biochemical Deterioration Context

- **Mass-Balance Fluid Conservation**: Non-negative fluid volume clamping ($0 \le V \le 1000\text{ L}$).
- **FIFO Batch Tracking**: Volume-weighted fluid retention age without artificial age resets.
- **Deterioration Index $\ge 0.50$**: Prompts `REVIEW_REQUIRED` or `PRIORITIZE_USE`.
- **Critical Stagnation ($\ge 36\text{ h}$ idle)**: Prompts `EMPTY_AND_REASSESS`.



## 8. Conflict Resolution & Precedence Arbitration

The `SmartRoutingEngine` resolves operational edge cases transparently:
- **ML = Irrigation, Weather = Heavy Rain** $\rightarrow$ Route: `Restricted Irrigation`, Action: `STORE_FOR_LATER`.
- **ML = Irrigation, Quality = Critical Pathogen, Weather = Sunny** $\rightarrow$ Route: `Sewer Bypass`, Action: `SAFETY_OVERRIDE`.
- **ML = Indoor Reuse, Anomaly = Flagged** $\rightarrow$ Route: `Indoor Reuse`, Status: `ANOMALY_REVIEW`.



## 9. Full Batch Evaluation on Holdout Test Dataset

We execute batch evaluation across the 225 holdout test samples (`dataset/processed/test.csv`).



In [ ]:
test_csv = os.path.join(project_root, "dataset", "processed", "test.csv")
out_csv = os.path.join(project_root, "dataset", "processed", "routing_decisions.csv")

df_decisions = pipeline.evaluate_batch(test_csv, output_csv_path=out_csv)
print(f"Evaluated {len(df_decisions)} holdout test samples.")
df_decisions[['Sample_ID', 'source', 'predicted_class_name', 'prediction_confidence', 'final_route', 'final_action', 'decision_status']].head(10)



## 10. Multi-Variable Scenario Analysis

We evaluate the 10 controlled synthetic scenarios demonstrating all multi-variable permutations.



In [ ]:
sim_csv = os.path.join(project_root, "simulation", "data", "routing_decision_simulation.csv")
df_scenarios = run_synthetic_routing_scenarios(output_path=sim_csv)

df_scenarios[['scenario_id', 'source', 'predicted_route', 'prediction_confidence', 'safety_status', 'weather_action', 'final_route', 'final_action', 'override_applied']]



## 11. Visualizations & Decision Analytics


In [ ]:
# Visualization 1: Final Route Distribution
plt.figure(figsize=(9, 4.5))
route_counts = df_decisions['final_route'].value_counts()
colors = ['#d62728', '#2ca02c', '#ff7f0e', '#1f77b4']
bars = plt.bar(route_counts.index, route_counts.values, color=colors, edgecolor='black', alpha=0.85)
plt.title('Figure 1: Final Operational Route Distribution on Holdout Test Set')
plt.xlabel('Operational Route')
plt.ylabel('Number of Samples')
for bar in bars:
    y = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, y + 2, f"{y} ({(y/len(df_decisions))*100:.1f}%)", ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()



In [ ]:
# Visualization 2: Override Distribution
plt.figure(figsize=(8, 4))
override_counts = df_decisions['override_applied'].value_counts()
labels = ['Model Route Retained (False)', 'Safety Override Applied (True)']
plt.bar(labels, [override_counts.get(False, 0), override_counts.get(True, 0)], color=['#2ca02c', '#d62728'], edgecolor='black', width=0.5)
plt.title('Figure 2: Safety Override Frequency on Holdout Test Set')
plt.ylabel('Sample Count')
plt.tight_layout()
plt.show()



In [ ]:
# Visualization 3: Anomaly Status vs Final Route
plt.figure(figsize=(10, 4.5))
ct = pd.crosstab(df_decisions['anomaly_status'], df_decisions['final_route'])
ct.plot(kind='bar', stacked=True, ax=plt.gca(), colormap='viridis', edgecolor='black', alpha=0.9)
plt.title('Figure 3: Anomaly Screening vs Final Route Assignment')
plt.xlabel('Anomaly Status')
plt.ylabel('Number of Samples')
plt.legend(title='Final Route')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()



In [ ]:
# Visualization 4: Prediction Confidence Distribution
plt.figure(figsize=(9, 4))
plt.hist(df_decisions['prediction_confidence'], bins=20, color='#1f77b4', edgecolor='black', alpha=0.75)
plt.axvline(0.75, color='green', linestyle='--', linewidth=1.5, label='High Confidence (0.75)')
plt.axvline(0.60, color='orange', linestyle='--', linewidth=1.5, label='Acceptable Confidence (0.60)')
plt.title('Figure 4: Model Prediction Confidence Distribution')
plt.xlabel('XGBoost Maximum Class Probability')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.show()



## 12. Safety Override Diagnostics

Examining samples where the independent safety layer overrode the machine learning candidate route.



In [ ]:
overrides_df = df_decisions[df_decisions['override_applied'] == True]
print(f"Total Safety Overrides in Test Set: {len(overrides_df)}")
if len(overrides_df) > 0:
    display_cols = ['Sample_ID', 'source', 'predicted_class_name', 'final_route', 'safety_status', 'reason_codes', 'human_readable_reason']
    print(overrides_df[display_cols].to_string())
else:
    print("Zero false-safe test errors required manual safety override in this clean test partition.")



## 13. Systemic & Scientific Limitations

1. **Decision-Support Scope**: The engine provides real-time operational routing recommendations, not regulatory compliance certification.
2. **Rule-Derived Ground Truth**: Models learn patterns from rule-derived operational labels benchmarked against EPA/WHO guidelines.
3. **Decoupled Telemetry**: Sensor readings represent single-point physical measurements; multi-point hydraulic stagnation requires spatial sensors.
4. **No Direct Pathogen Assay**: Unsupervised anomalies reflect statistical variance, not direct microbial viability.



## 14. Conclusions & Phase 12 Readiness

- **Central Smart Routing Engine** successfully deployed in `routing/routing_engine.py`.
- Strict priority hierarchy verified across 93 unit tests:
  - Favorable weather **CANNOT** override critical safety.
  - Anomaly alone **CANNOT** force Sewer Bypass.
  - Irrigation is deferred during rainfall while preserving route class.
  - Indoor reuse remains hydraulically decoupled from ambient rain.
- **The system is fully prepared for Phase 12 (SHAP Explainability & Feature Importance).**

